# 05 · Compare exp01 (fixed spawn) vs exp02 (randomized spawn)

Both experiments sweep the same grid (`step_mul` × `aggression`) with
`AttackNearestPolicy` in `5m_vs_5m`. The only difference is that exp02 samples
a new distance and bearing between the two groups every episode (see
`envs.spawn_sampling.sample_opposing_spawns`), instead of the fixed, mirrored
spawn points of exp01. Everything here reuses the sweep helpers from `04` (now
in `sindy_wm.evaluation.sweep`), so this notebook only adds the side-by-side
comparison.

If exp02 is still being collected, `load_all_indexes` reports and skips any
dataset that isn't `complete` yet — rerun this notebook once collection
finishes for the full 4×4 grid.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sindy_wm.evaluation.sweep import heatmap, load_all_indexes, summarize
from sindy_wm.paths import RAW_DIR

plt.rcParams["figure.dpi"] = 100

EXPERIMENTS = {
    "exp01 (fixed spawn)": RAW_DIR / "exp01_5m_aggression_stepmul",
    "exp02 (randomized spawn)": RAW_DIR / "exp02_5m_random_position_aggression_stepmul",
}

## 1 · Load and summarize both experiments

Same `load_all_indexes` + `summarize` pipeline as notebook 04, Part 1, run once per experiment.

In [ ]:
summaries = {}
for label, experiment_dir in EXPERIMENTS.items():
    episodes = load_all_indexes(experiment_dir)
    summaries[label] = (
        episodes.groupby(["dataset", "step_mul", "aggression"])
        .apply(summarize, include_groups=False)
        .reset_index()
        .sort_values(["step_mul", "aggression"], ignore_index=True)
    )
    print(f"{label}: {episodes['dataset'].nunique()} datasets, {len(episodes)} episodes")

## 2 · Win rate: side by side, and the difference

If win rate looks broadly similar between the two, the policy/step_mul effects from exp01 carry
over once spawn distance varies; the third panel makes any shift explicit (positive = exp02 wins
more at that setting).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, s) in zip(axes, summaries.items(), strict=True):
    heatmap(s, "win_rate", label, ax=ax)
fig.tight_layout()
plt.show()

exp01_summary, exp02_summary = summaries.values()
merged = exp01_summary.merge(
    exp02_summary, on=["step_mul", "aggression"], how="outer", suffixes=("_exp01", "_exp02")
)
merged["win_rate_delta"] = merged["win_rate_exp02"] - merged["win_rate_exp01"]

diff_table = merged.pivot(index="aggression", columns="step_mul", values="win_rate_delta")
lim = float(np.nanmax(np.abs(diff_table.to_numpy()))) or 1.0
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(diff_table, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(diff_table.columns)), diff_table.columns)
ax.set_yticks(range(len(diff_table.index)), diff_table.index)
ax.set_xlabel("step_mul")
ax.set_ylabel("aggression")
ax.set_title("Win rate: exp02 − exp01")
for i, a in enumerate(diff_table.index):
    for j, st in enumerate(diff_table.columns):
        v = diff_table.loc[a, st]
        if pd.notna(v):
            ax.text(j, i, f"{v:+.0%}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Δ win rate")
plt.show()

## 3 · Did randomizing spawn actually make episodes more varied?

This is the thing exp02 set out to test. `distinct_outcomes` counts how many different
(length, final health) combinations occur in a dataset — exp01 had several settings at or near 1
(aggression 1.0 especially: see the README's findings), meaning essentially one game repeated.

In [ ]:
step_muls = sorted(merged["step_mul"].dropna().unique())
x = np.arange(len(step_muls))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
for offset, (label, s) in zip([-width / 2, width / 2], summaries.items(), strict=True):
    means = s.groupby("step_mul")["distinct_outcomes"].mean().reindex(step_muls)
    ax.bar(x + offset, means, width, label=label)
ax.set_xticks(x, step_muls)
ax.set_xlabel("step_mul")
ax.set_ylabel("distinct outcomes (mean across aggressions, out of n=100 or 500 episodes)")
ax.set_title("Episode variety: fixed vs. randomized spawn")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.show()

## 4 · Battle duration

Randomized spawn distance (8–16 map units, vs. a fixed 14) should widen the approach phase a bit;
this checks whether that shows up in mean duration once averaged over aggression.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, s in summaries.items():
    means = s.groupby("step_mul")["duration_mean_s"].mean().reindex(step_muls)
    ax.plot(step_muls, means, marker="o", label=label)
ax.set_xlabel("step_mul")
ax.set_ylabel("mean duration (s)")
ax.set_title("Battle duration: fixed vs. randomized spawn")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5 · Full side-by-side table

Every setting, both experiments, with the win-rate delta. `NaN` in an `_exp02` column means that
dataset hasn't finished collecting yet.

In [ ]:
cols = [
    "step_mul",
    "aggression",
    "win_rate_exp01",
    "win_rate_exp02",
    "win_rate_delta",
    "distinct_outcomes_exp01",
    "distinct_outcomes_exp02",
    "duration_mean_s_exp01",
    "duration_mean_s_exp02",
]
merged.sort_values(["step_mul", "aggression"])[cols].style.format(
    {
        "win_rate_exp01": "{:.0%}",
        "win_rate_exp02": "{:.0%}",
        "win_rate_delta": "{:+.0%}",
        "duration_mean_s_exp01": "{:.1f}",
        "duration_mean_s_exp02": "{:.1f}",
    }
)

## Next steps

- Once exp02 finishes, re-run this and update the README's findings section with the comparison.
- A per-dataset SINDy coefficient comparison (do the fitted equations change between fixed and
  randomized spawn?) is a natural follow-up, but `save_run`'s `config.json` doesn't currently
  record *which experiment* a dataset came from — only the bare dataset name, which exp01 and
  exp02 both reuse. Worth adding `"experiment": EXPERIMENT` to `run_config["dataset"]` in notebook
  04 before relying on saved runs to tell the two apart.
- If spawn distance turns out to matter a lot, it's recoverable per episode from the step-0
  snapshot (`x`, `y` per unit) even though it isn't in `episodes.jsonl` — could be added as a
  column in Part 1's `episodes` table here, to stratify win rate by initial distance directly.